# Geospatial Analysis with DuckDB and Mapbox

In this notebook we'll answer a simple question: **which DC elementary schools are the most walkable?**

We'll measure walkability as the number of addresses within a 10-minute walk of each school, using:
- **DuckDB** (with its spatial extension) — a fast analytical database that understands geometry
- **Mapbox Isochrone API** — generates polygons representing areas reachable within a travel-time threshold
- **Mapbox GL JS** — renders an interactive map of the results

## 1. Dependencies

When you ran `uv sync`, it installed everything we need:
- `duckdb` — our geospatial database
- `requests` — for HTTP calls to the Mapbox API
- `jupyter` — to run this notebook

No `pip install` needed.

## 2. Credentials

Get a free access token at [account.mapbox.com](https://account.mapbox.com). Replace the placeholder below with your token.

In [16]:
MAPBOX_ACCESS_TOKEN = "pk.eyJ1Ijoic2JtYTQ0IiwiYSI6ImNtbjZmanRieTA1NWkyc29mamFlcXhoOGgifQ.naA2YBz6btj9chkdeeuI0A"

## 3. Download the Data

We need two datasets. Create a `data/` folder at the root of this repo and save the files there.

### DC Public Schools
1. Go to [opendata.dc.gov/datasets/dc-public-schools](https://opendata.dc.gov/datasets/dc-public-schools/about)
2. Click **Download** → select **GeoJSON**
3. Save as `data/DC_Public_Schools.geojson`

### DC Address Points
1. Go to [opendata.dc.gov/pages/addressing-in-dc#data](https://opendata.dc.gov/pages/addressing-in-dc#data)
2. Download the **Address Points** shapefile
3. Unzip into `data/Address_Points/` so you have `data/Address_Points/Address_Points.shp` (and `.dbf`, `.shx`, etc.)

Your `data/` folder should look like:
```
data/
  DC_Public_Schools.geojson
  Address_Points/
    Address_Points.shp
    Address_Points.dbf
    Address_Points.prj
    Address_Points.shx
```

Examine the files in QGIS to ensure we're looking at the right thing. Do we understand how to filter the dataset? You may need to go looking for [additional documentation](https://opendata.dc.gov/documents/130778ae88bb433cb0024298c478ab46/explore).

## 4. Load Data into DuckDB

DuckDB's spatial extension can read GeoJSON and shapefiles directly with `ST_Read`. We'll load both datasets into a local database file.

In [17]:
import duckdb

conn = duckdb.connect("dc_schools.duckdb", config={"storage_compatibility_version": "v1.5.0"})

conn.execute("INSTALL spatial")
conn.execute("LOAD spatial")

conn.execute("""
    CREATE OR REPLACE TABLE schools AS
    SELECT * FROM ST_Read('data/DC_Public_Schools.geojson')
""")

# The address shapefile is in EPSG:3857 (Web Mercator).
# ST_Transform to EPSG:4326 returns (lat, lon) axis order per the EPSG spec,
# so we flip to (lon, lat) for compatibility with GeoJSON and Mapbox.
conn.execute("""
    CREATE OR REPLACE TABLE addresses AS
    SELECT * EXCLUDE (geom),
           ST_FlipCoordinates(ST_Transform(geom, 'EPSG:3857', 'EPSG:4326')) AS geom
    FROM ST_Read('data/Address_Points/Address_Points.shp')
""")

schools_count = conn.execute("SELECT COUNT(*) FROM schools").fetchone()[0]
addresses_count = conn.execute("SELECT COUNT(*) FROM addresses").fetchone()[0]
print(f"Loaded {schools_count} schools and {addresses_count} addresses")

Loaded 123 schools and 144372 addresses


In [18]:
print("=== schools ===")
for row in conn.execute("DESCRIBE schools").fetchall():
    print(row)

print("\n=== addresses ===")
for row in conn.execute("DESCRIBE addresses").fetchall():
    print(row)

=== schools ===
('OGC_FID', 'BIGINT', 'YES', None, None, None)
('NAME', 'VARCHAR', 'YES', None, None, None)
('ADDRESS', 'VARCHAR', 'YES', None, None, None)
('FACUSE', 'VARCHAR', 'YES', None, None, None)
('LEVEL_', 'VARCHAR', 'YES', None, None, None)
('STATUS', 'VARCHAR', 'YES', None, None, None)
('PHONE', 'VARCHAR', 'YES', None, None, None)
('TOTAL_STUD', 'INTEGER', 'YES', None, None, None)
('SSL', 'VARCHAR', 'YES', None, None, None)
('GIS_ID', 'VARCHAR', 'YES', None, None, None)
('WEB_URL', 'VARCHAR', 'YES', None, None, None)
('BLDG_NUM', 'INTEGER', 'YES', None, None, None)
('SCH_PROG', 'VARCHAR', 'YES', None, None, None)
('CAPITALGAINS', 'VARCHAR', 'YES', None, None, None)
('CAPACITY', 'INTEGER', 'YES', None, None, None)
('YEAR_BUILT', 'INTEGER', 'YES', None, None, None)
('SQUARE_FOOTAGE', 'INTEGER', 'YES', None, None, None)
('POPULATION_PLAN', 'INTEGER', 'YES', None, None, None)
('LONGITUDE', 'DOUBLE', 'YES', None, None, None)
('LATITUDE', 'DOUBLE', 'YES', None, None, None)
('SCHOOL

In [19]:
print("=== first 3 schools ===")
for row in conn.execute("SELECT * FROM schools LIMIT 3").fetchall():
    print(row)

print("\n=== distinct school levels ===")
for row in conn.execute("SELECT DISTINCT LEVEL_ FROM schools ORDER BY LEVEL_").fetchall():
    print(row[0])

=== first 3 schools ===
(0, 'Key Elementary School', '5001 DANA PLACE NW', 'Elementary School', 'ES', 'Active', '202-729-3280', 386, ' ', 'dcps_272', 'http://keyschooldc.org/dcps/', 272, '272', 'N', None, 1929, None, None, -77.10052732, 38.92680184, '2025-2026', 'District of Columbia Public Schools', 1, 'Key Elementary School', 272, 'PK4-5th', 294605, 391283.15, 139885.29, 20016, 'DGS_0653.0', '{18A1A4C0-A3F1-4C71-8AE0-95503BC34190}', datetime.datetime(2025, 2, 13, 21, 20, 34, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>), datetime.datetime(2025, 8, 25, 18, 47, 39, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>), 1601, b'\x01\x01\x00\x00\x00\x87v\xcc\x13oFS\xc0\xcd\xa4\xda\xaf\xa1vC@')
(1, 'King Elementary School', '3200 6TH STREET SE', 'Elementary School', 'ES', 'Active', '202-939-4900', 345, ' ', 'dcps_344', 'http://profiles.dcps.dc.gov/King+Elementary+School', 344, '344', 'N', 525, 1971, 65500, 400, -76.99823899, 38.84244319, '2025-2026', 'District of

### 4. Visualize 10,000 random address points
The full dataset is hundreds of thousands of points--a lot to stick in browser memory all at once.

In [1]:
import json
import base64
from IPython.display import IFrame

sample = conn.execute("""
    SELECT ST_X(geom) AS lng, ST_Y(geom) AS lat
    FROM addresses
    ORDER BY RANDOM()
    LIMIT 10000
""").fetchall()

# Print a few raw coordinates to check they look like lat/lng
print("Sample coordinates:", sample[:3])

geojson = json.dumps({
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": {}
        }
        for lng, lat in sample
    ]
})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/light-v11',
    center: [-77.0369, 38.9072],
    zoom: 12
  });
  map.on('load', () => {
    map.addSource('addresses', { type: 'geojson', data: GEOJSON_DATA });
    map.addLayer({
      id: 'addresses',
      type: 'circle',
      source: 'addresses',
      paint: { 'circle-radius': 3, 'circle-color': '#e55e5e', 'circle-opacity': 0.6 }
    });
  });
</script>
</body>
</html>"""

html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN).replace('GEOJSON_DATA', geojson)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="480px")

NameError: name 'conn' is not defined

## 5. Visualize Schools

In [21]:
rows = conn.execute("""
    SELECT NAME, ST_X(geom) AS lng, ST_Y(geom) AS lat
    FROM schools
""").fetchall()

geojson = json.dumps({
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": {"name": name}
        }
        for name, lng, lat in rows
    ]
})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/light-v11',
    center: [-77.0369, 38.9072],
    zoom: 11
  });
  map.on('load', () => {
    map.addSource('schools', { type: 'geojson', data: GEOJSON_DATA });
    map.addLayer({
      id: 'schools',
      type: 'circle',
      source: 'schools',
      paint: { 'circle-radius': 5, 'circle-color': '#3887be', 'circle-stroke-width': 1, 'circle-stroke-color': '#fff' }
    });
    map.on('click', 'schools', (e) => {
      new mapboxgl.Popup().setLngLat(e.lngLat).setHTML(e.features[0].properties.name).addTo(map);
    });
    map.on('mouseenter', 'schools', () => { map.getCanvas().style.cursor = 'pointer'; });
    map.on('mouseleave', 'schools', () => { map.getCanvas().style.cursor = ''; });
  });
</script>
</body>
</html>"""

html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN).replace('GEOJSON_DATA', geojson)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="480px")

## 6. Generate Isochrones

An **isochrone** is a polygon representing everything reachable from a point within a given travel time. You can experiment with creating them on the [Mapbox Isochrone Playground](https://docs.mapbox.com/playground/isochrone/).

We'll call the [Mapbox Isochrone API](https://docs.mapbox.com/api/navigation/isochrone/) to get 10-minute walking polygons for each elementary school.

The API endpoint format is:
```
GET https://api.mapbox.com/isochrone/v1/mapbox/walking/{lng},{lat}
    ?contours_minutes=10&polygons=true&access_token={token}
```

We'll save each response as a GeoJSON file in the `isochrones/` folder. If you run this cell again, it skips schools that already have a file (so you can resume if interrupted).

In [22]:
import os
import re
import json
import time
import requests

os.makedirs("isochrones", exist_ok=True)

def safe_filename(name):
    """Convert a school name to a safe filename."""
    return re.sub(r'[^\w]', '_', name)

# Filter to elementary schools — adjust the value if LEVEL_ uses a different label
elementary_schools = conn.execute("""
    SELECT NAME, ST_X(geom) AS lng, ST_Y(geom) AS lat
    FROM schools
    WHERE LEVEL_ = 'ES'
""").fetchall()

print(f"Found {len(elementary_schools)} elementary schools")

for name, lng, lat in elementary_schools:
    filepath = f"isochrones/{safe_filename(name)}.geojson"

    if os.path.exists(filepath):
        print(f"  skip  {name}")
        continue

    url = (
        f"https://api.mapbox.com/isochrone/v1/mapbox/walking/{lng},{lat}"
        f"?contours_minutes=10&polygons=true&access_token={MAPBOX_ACCESS_TOKEN}"
    )
    response = requests.get(url)
    response.raise_for_status()

    data = response.json()
    data["school_name"] = name  # tag the file with the school name for easy reload

    with open(filepath, "w") as f:
        json.dump(data, f)

    print(f"  saved {name}")
    time.sleep(0.1)  # stay well within API rate limits

print("Done!")

Found 74 elementary schools
  skip  Key Elementary School
  skip  King Elementary School
  skip  Lafayette Elementary School
  skip  Langdon Elementary School
  skip  Langley Elementary School
  skip  LaSalle-Backus Elementary School
  skip  Leckie Education Campus
  skip  Ludlow-Taylor Elementary School
  skip  Malcolm X Elementary School @ Green
  skip  Lorraine H. Whitlock Elementary School
  skip  Mann Elementary School
  skip  Marie Reed Elementary School
  skip  Miner Elementary School
  skip  Moten Elementary School
  skip  Murch Elementary School
  skip  Nalle Elementary School
  skip  Noyes Elementary School
  skip  Oyster-Adams Bilingual School (Oyster)
  skip  Patterson Elementary School
  skip  Payne Elementary School
  skip  Peabody Elementary School
  skip  Plummer Elementary School
  skip  Powell Elementary School
  skip  Randle Highlands Elementary School
  skip  Raymond Elementary School
  skip  Kimball Elementary School
  skip  Hyde-Addison Elementary School
  skip  M

## 7. Load Isochrones into DuckDB

We'll add an `isochrone` geometry column to the `schools` table and populate it from the GeoJSON files.

In [3]:
import glob

conn.execute("ALTER TABLE schools ADD COLUMN IF NOT EXISTS isochrone GEOMETRY")

for filepath in glob.glob("isochrones/*.geojson"):
    with open(filepath) as f:
        data = json.load(f)

    school_name = data.get("school_name")
    if not school_name:
        continue

    # GeoJSON features wrap geometries, and allow them to be paired with extra data ("properties").
    # Here, we just want the geometry from each isochrone feature.
    geometry = json.dumps(data["features"][0]["geometry"])

    conn.execute(
        "UPDATE schools SET isochrone = ST_GeomFromGeoJSON(?) WHERE NAME = ?",
        [geometry, school_name]
    )

loaded = conn.execute("SELECT COUNT(*) FROM schools WHERE isochrone IS NOT NULL").fetchone()[0]
print(f"Loaded isochrones for {loaded} schools")

NameError: name 'conn' is not defined

## 8. Visualize Schools with Isochrones

In [24]:
rows = conn.execute("""
    SELECT NAME, ST_X(geom) AS lng, ST_Y(geom) AS lat, ST_AsGeoJSON(isochrone) AS iso
    FROM schools
    WHERE isochrone IS NOT NULL
""").fetchall()

features = []
for name, lng, lat, iso in rows:
    features.append({
        "type": "Feature",
        "geometry": json.loads(iso),
        "properties": {"name": name}
    })
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lng, lat]},
        "properties": {"name": name}
    })

geojson = json.dumps({"type": "FeatureCollection", "features": features})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/light-v11',
    center: [-77.0369, 38.9072],
    zoom: 11
  });
  map.on('load', () => {
    map.addSource('data', { type: 'geojson', data: GEOJSON_DATA });
    map.addLayer({
      id: 'isochrones',
      type: 'fill',
      source: 'data',
      filter: ['==', ['geometry-type'], 'Polygon'],
      paint: { 'fill-color': '#3887be', 'fill-opacity': 0.15 }
    });
    map.addLayer({
      id: 'isochrones-outline',
      type: 'line',
      source: 'data',
      filter: ['==', ['geometry-type'], 'Polygon'],
      paint: { 'line-color': '#3887be', 'line-width': 1.5 }
    });
    map.addLayer({
      id: 'schools',
      type: 'circle',
      source: 'data',
      filter: ['==', ['geometry-type'], 'Point'],
      paint: { 'circle-radius': 5, 'circle-color': '#e55e5e', 'circle-stroke-width': 1, 'circle-stroke-color': '#fff' }
    });
    map.on('click', 'schools', (e) => {
      new mapboxgl.Popup().setLngLat(e.lngLat).setHTML(e.features[0].properties.name).addTo(map);
    });
    map.on('mouseenter', 'schools', () => { map.getCanvas().style.cursor = 'pointer'; });
    map.on('mouseleave', 'schools', () => { map.getCanvas().style.cursor = ''; });
  });
</script>
</body>
</html>"""

html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN).replace('GEOJSON_DATA', geojson)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="480px")

## 9. Walkability Analysis

Now we run a **spatial join**: for each address point, we check whether it falls inside any school's isochrone polygon. DuckDB evaluates this with `ST_Within` and groups the results by school.

In [25]:
conn.execute("""
    CREATE OR REPLACE TABLE walkability AS
    SELECT
        schools.NAME AS school_name,
        COUNT(addresses.geom) AS address_count
    FROM   schools
    JOIN   addresses ON ST_Within(addresses.geom, schools.isochrone)
    GROUP  BY schools.NAME
    ORDER  BY address_count DESC
""")

for row in conn.execute("SELECT school_name, address_count FROM walkability").fetchall():
    print(f"{row[1]:>6}  {row[0]}")

  5701  Maury Elementary School
  5099  School-Within-School
  4445  Payne Elementary School
  4389  Peabody Elementary School
  4225  Ludlow-Taylor Elementary School
  4111  Watkins Elementary School
  4103  Tubman Elementary School
  3693  J.O. Wilson Elementary School
  3615  Raymond Elementary School
  3613  Miner Elementary School
  3607  Barnard Elementary School
  3587  Shirley Chisholm Elementary School
  3471  Garrison Elementary School
  3265  Truesdell Elementary School
  3255  Cleveland Elementary School
  3243  Brent Elementary School
  3193  Hyde-Addison Elementary School
  3173  Ross Elementary School
  3146  Langley Elementary School
  3115  Bruce-Monroe Elementary School @ Park View
  2935  Marie Reed Elementary School
  2561  Seaton Elementary School
  2438  Nalle Elementary School
  2395  H.D. Cooke Elementary School
  2374  Whittier Elementary School
  2363  Capitol Hill Montessori
  2327  Dorothy I. Height Elementary School
  2261  Bancroft Elementary School
  2053

## Interactive Map

Each school is shown as a circle. **Larger circles = more addresses within a 10-minute walk.** Click any circle to see the school name and address count.

In [34]:
rows = conn.execute("""
    SELECT walkability.school_name, ST_X(schools.geom) AS lng, ST_Y(schools.geom) AS lat, walkability.address_count
    FROM walkability
    JOIN schools ON walkability.school_name = schools.SCHOOL_NAM
""").fetchall()

geojson = json.dumps({
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": {"name": name, "address_count": count}
        }
        for name, lng, lat, count in rows
    ]
})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/light-v11',
    center: [-77.0369, 38.9072],
    zoom: 11
  });
  map.on('load', () => {
    map.addSource('schools', { type: 'geojson', data: GEOJSON_DATA });
    map.addLayer({
      id: 'schools',
      type: 'circle',
      source: 'schools',
      paint: {
        'circle-radius': ['interpolate', ['linear'], ['get', 'address_count'], 0, 4, 1000, 22],
        'circle-color': '#3887be',
        'circle-opacity': 0.85,
        'circle-stroke-width': 1.5,
        'circle-stroke-color': '#ffffff'
      }
    });
    map.on('click', 'schools', (e) => {
      const p = e.features[0].properties;
      new mapboxgl.Popup()
        .setLngLat(e.lngLat)
        .setHTML('<strong>' + p.name + '</strong><br>' + p.address_count + ' addresses within 10-min walk')
        .addTo(map);
    });
    map.on('mouseenter', 'schools', () => { map.getCanvas().style.cursor = 'pointer'; });
    map.on('mouseleave', 'schools', () => { map.getCanvas().style.cursor = ''; });
  });
</script>
</body>
</html>"""

html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN).replace('GEOJSON_DATA', geojson)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="520px")